# 2. Data Splitting

Before training a model, your dataset must be divided into **training**,
**validation**, and **test** subsets.  How you split matters — a well-chosen
split tests whether the model can *generalise*, not just memorise.

**What you will do:**
1. Load the molecules from your cleaned dataset from notebook 1
2. Random split — the simplest baseline
3. Scaffold split — a more challenging, chemistry-aware split
4. Metric-based split — split by any molecular property or distance metric
5. Compare the three strategies visually and statistically

**Core function:** `split_by_values(molecules, values, ratio, high_values_in_test)`

All splitting strategies share the same logic: sort molecules by a numeric
value and assign the bottom fraction to train, the next to validation, and
the top fraction to test.  The choice of *values* determines what ends up
in the test set.

See `docs/metrics_reference.md` §6 for the mathematical description.

## Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from evaluation import load_smiles
from evaluation.metrics import nearest_neighbor_distance
from evaluation.properties import compute_properties
from evaluation.splitting import novelty, random_split, scaffold_split, split_by_values
from evaluation.visualization import (
    plot_distribution_comparison,
    compare_distributions,
)

print("Imports OK")

In [ ]:
# Set the base directory to the current working directory (where this notebook is located)
base_dir = os.path.dirname(os.path.abspath("__file__"))

## Load Molecules

In [ ]:
# Load the SMILES from your file
csv_path = os.path.join(base_dir, "your_path_here")                 # Please insert the path to your CSV file here

SMILES = load_smiles(csv_path)

print(f"Loaded {len(SMILES)} molecules.")

---
## Strategy 1 — Random Split

Molecules are assigned to train / validation / test at random.

This is the simplest baseline — train and test sets should have very similar property distributions.

In [ ]:
train_random, val_random, test_random = random_split(SMILES, ratio=(0.8, 0.1, 0.1), seed=42)

print(f"Train : {len(train_random)} molecules")
print(f"Val   : {len(val_random)} molecules")
print(f"Test  : {len(test_random)} molecules")

In [ ]:
# ── Compare property distributions: train vs. test ───────────────────────────
props_train_r = compute_properties(train_random)
props_test_r  = compute_properties(test_random)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, prop in zip(axes, ["molecular_weight", "logp", "topological_polar_surface_area"]):
    plot_distribution_comparison(
        props_train_r[prop], props_test_r[prop],
        labels=["Train", "Test"],
        xlabel=prop,
        ax=ax
    )
plt.suptitle("Random split — property distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# How structurally different are train and test?
nnd_random = nearest_neighbor_distance(test_random, reference=train_random)
print(f"Nearest-neighbor distance (test → train), random split: {nnd_random:.3f}")
print("A low value means test molecules are structurally close to training molecules.")

---
## Strategy 2 — Scaffold Split

Molecules with **rare scaffolds** are assigned to the test set; molecules with
**common scaffolds** stay in training.  This forces the model to generalise to
ring systems it has seen less often — a more realistic evaluation of
generalisation ability.

In [ ]:
train_scaffold, val_scaffold, test_scaffold = scaffold_split(SMILES, ratio=(0.8, 0.1, 0.1), seed=42)

print(f"Train : {len(train_scaffold)} molecules")
print(f"Val   : {len(val_scaffold)} molecules")
print(f"Test  : {len(test_scaffold)} molecules")

In [ ]:
# ── Compare property distributions: train vs. test ───────────────────────────
props_train_s = compute_properties(train_scaffold)
props_test_s  = compute_properties(test_scaffold)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, prop in zip(axes, ["molecular_weight", "logp", "topological_polar_surface_area"]):
    plot_distribution_comparison(
        props_train_s[prop], props_test_s[prop],
        labels=["Train", "Test"],
        xlabel=prop,
        ax=ax
    )
plt.suptitle("Scaffold split — property distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
nnd_scaffold = nearest_neighbor_distance(test_scaffold, reference=train_scaffold)
print(f"Nearest-neighbor distance (test → train), scaffold split: {nnd_scaffold:.3f}")
print(f"Nearest-neighbor distance (test → train), random split:   {nnd_random:.3f}")

Which split produces a more challenging test set? Why?

---
## Strategy 3 — Metric-based Split

Any per-molecule property or metric can be used as the splitting criterion by
calling `split_by_values` directly.  This is the plug-and-play interface: you
supply a list of molecules and a parallel list of numeric values, and the
function handles the rest.

### The `high_values_in_test` flag

The flag controls *which end* of the sorted list becomes the test set:

| `high_values_in_test` | Sort order | Test set contains |
|---|---|---|
| `True` (default) | ascending | molecules with the **highest** values |
| `False` | descending | molecules with the **lowest** values |

**Example — splitting by molecular weight:**
- `high_values_in_test=True` → the **heaviest** molecules go to the test set.
  Use this to ask: *can the model generalise to larger molecules than it trained on?*
- `high_values_in_test=False` → the **lightest** molecules go to the test set.
  Use this to ask: *can the model generalise to smaller, simpler molecules?*

The same logic applies to any other property: LogP, TPSA, QED, or any custom
metric you compute.  Choose the direction that best matches your scientific question. 
For more information on which metrics we have provided in the `evaluation` package, see [the metrics reference guide](./docs/metrics_reference.md).

In [ ]:
# ── Compute properties for the full set ───────────────────────────────────────
props_all = compute_properties(SMILES)

# Choose the splitting criterion: molecular weight
# The SMILES list and the values list must be in the same order.
smiles_list = list(props_all.index)          # canonical SMILES (the DataFrame index)
mw_values   = props_all["molecular_weight"].tolist()

# Split: heaviest molecules → test  (high_values_in_test=True, the default)
train_mw, val_mw, test_mw = split_by_values(
    smiles_list, mw_values, ratio=(0.8, 0.1, 0.1), high_values_in_test=True
)

print(f"Train : {len(train_mw)} molecules")
print(f"Val   : {len(val_mw)} molecules")
print(f"Test  : {len(test_mw)} molecules")

In [ ]:
# ── Compare property distributions: train vs. test ───────────────────────────
props_train_mw = compute_properties(train_mw)
props_test_mw  = compute_properties(test_mw)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, prop in zip(axes, ["molecular_weight", "logp", "topological_polar_surface_area"]):
    plot_distribution_comparison(
        props_train_mw[prop], props_test_mw[prop],
        labels=["Train", "Test"],
        xlabel=prop,
        ax=ax
    )
plt.suptitle("MW-based split — property distributions", y=1.02)
plt.tight_layout()
plt.show()
# Note: the molecular weight panel should look clearly different for train vs. test!

In [ ]:
nnd_mw = nearest_neighbor_distance(test_mw, reference=train_mw)
print(f"Nearest-neighbor distance (test → train), MW split:      {nnd_mw:.3f}")
print(f"Nearest-neighbor distance (test → train), scaffold split: {nnd_scaffold:.3f}")
print(f"Nearest-neighbor distance (test → train), random split:   {nnd_random:.3f}")

---
## Statistical Comparison of Splits

Use `compare_distributions` to formally test whether property distributions
differ between train and test for each split strategy.

In [ ]:
# ── KS test for each split on three properties ───────────────────────────────
properties_to_test = ["molecular_weight", "logp", "topological_polar_surface_area"]

rows = []
for prop in properties_to_test:
    for split_name, props_tr, props_te in [
        ("random",   props_train_r,  props_test_r),
        ("scaffold", props_train_s,  props_test_s),
        ("MW-based", props_train_mw, props_test_mw),
    ]:
        result = compare_distributions(props_tr[prop], props_te[prop], test="ks")
        rows.append({"property": prop, "split": split_name,
                     "statistic": round(result["statistic"], 3),
                     "p_value":   round(result["p_value"], 3),
                     "significant": result["significant"]})

pd.DataFrame(rows)

---
## Student Challenges

---

### Challenge 1 — Design your own split

Pick a property other than molecular weight — for example, LogP, TPSA, or the
QED score — and use `split_by_values` to create a property-based split.

- Try `high_values_in_test=True` and `high_values_in_test=False`.  Which
  produces a more challenging test set according to nearest-neighbor distance?
- Compare the statistical tests for your split vs. the random split.

In [ ]:
# TODO: choose a property, compute per-molecule values, call split_by_values
#
# property_values = props_all["logp"].tolist()  # example: split by LogP
# train_custom, val_custom, test_custom = split_by_values(
#     smiles_list, property_values, ratio=(0.8, 0.1, 0.1), high_values_in_test=True
# )

### Challenge 2 — Split by nearest-neighbor distance

Use `nearest_neighbor_distance` to assign each molecule a value equal to its
distance from the rest of the set (internal NND).  Then use this as the
splitting criterion: the most structurally isolated molecules go to the test
set.

- What does this test set look like structurally?  Draw a molecule grid.
- How does the NND (test → train) compare to the scaffold split?

In [ ]:
# TODO: compute per-molecule nearest-neighbor distances and split by them
# Hint: you need the distance for each individual molecule, not the mean.
# Consider computing pairwise distances manually or using a loop.

### Challenge 3 — Evaluate novelty across splits

For each split strategy (random, scaffold, MW-based), compute:

1. `novelty(test, reference=train)` — at the canonical SMILES level
2. The nearest-neighbor distance (test → train)

Discuss: which split produces the most structurally novel test set?
Is a novel test set always better for evaluation?

In [ ]:
# TODO: compute and compare novelty and NND for each split strategy

### Challenge 4 — Save the Outputs

Based on your observations in the previous excercises, choose a splitting strategy and save the three different splits to individual files.



In [ ]:
# TODO: edit which train / val / test sets to use and update the output directory
train = train_random
val = val_random
test = test_random

output_dir = os.path.join(base_dir, "your_path_here")

# We write the SMILES to a file like this:
with open(f"{output_dir}/train.smi", "w") as f:
    [f.write(f"{smi}\n") for smi in train]

with open(f"{output_dir}/val.smi", "w") as f:
    [f.write(f"{smi}\n") for smi in val]

with open(f"{output_dir}/test.smi", "w") as f:
    [f.write(f"{smi}\n") for smi in test]
